# Algorithmic Reasoning Training for Nemotron-3Instead of teaching the model to COPY answers (SFT ceiling ~0.65),this trains the model to EXECUTE algorithms step-by-step.Each training example shows the ACTUAL solver computation:- Bit: XOR mask calculation, rotation verification- Gravity: g = 2d/t² for each example, then apply- Unit: factor = output/input, verify, apply- Cipher: character-by-character substitution table- Roman: step-by-step numeral constructionThe model learns to RUN the algorithm, not memorize outputs.

In [ ]:
# OFFLINE INSTALL - safe numpy handlingimport subprocess, sys# Try importing first - Kaggle might already have thesetry:    import datasets    import trl    from trl import SFTTrainer, SFTConfig    print("Already available - datasets:", datasets.__version__, "trl:", trl.__version__)except ImportError:    # Install ONLY if missing, without overwriting numpy    subprocess.check_call([        sys.executable, "-m", "pip", "install", "-q",        "--no-index",        "--find-links", "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages",        "datasets", "trl"    ], stderr=subprocess.DEVNULL)    import datasets, trl    from trl import SFTTrainer, SFTConfig    print("Installed - datasets:", datasets.__version__, "trl:", trl.__version__)

In [ ]:
# IMPORTSimport os, sys, stat, shutil, gc, zipfile, glob, re, typesimport polars as plimport torchimport torch.nn.functional as Fimport kagglehubfrom datasets import Datasetfrom peft import LoraConfig, get_peft_model, TaskType# Harden transformers import against broken sklearn/scipy in Kaggle runtime.import transformers.utils.import_utils as _tfufor _flag in (    "_sklearn_available", "_scipy_available",    "_vision_available", "_pil_available",    "_cv2_available", "_torchvision_available",):    if hasattr(_tfu, _flag):        setattr(_tfu, _flag, False)_tfu.is_sklearn_available = lambda: False_tfu.is_scipy_available = lambda: False_tfu.is_vision_available = lambda: False# Stub broken sklearn before importing transformerstry:    from sklearn.metrics import roc_curveexcept Exception:    for _mn in [m for m in list(sys.modules.keys()) if m == "sklearn" or m.startswith("sklearn.")]:        del sys.modules[_mn]    _sk = types.ModuleType("sklearn")    _sk.__path__ = []    _sk_m = types.ModuleType("sklearn.metrics")    _sk_m.roc_curve = lambda *a, **k: (_ for _ in ()).throw(RuntimeError("sklearn disabled"))    sys.modules["sklearn"] = _sk    sys.modules["sklearn.metrics"] = _sk_m# NOW import transformers after the patchesfrom transformers import AutoModelForCausalLM, AutoTokenizerfrom trl import SFTTrainer, SFTConfigos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")torch.backends.cuda.matmul.allow_tf32 = Truetry:    torch.backends.cuda.enable_flash_sdp(True)    torch.backends.cuda.enable_mem_efficient_sdp(True)except Exception:    pass

In [ ]:
# TRITON FIXdef _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,                     group_size=None, norm_before_gate=True, upcast=True):    dtype = x.dtype    if upcast: x = x.float()    variance = x.pow(2).mean(-1, keepdim=True)    x_normed = x * torch.rsqrt(variance + eps)    out = x_normed * weight.float()    if bias is not None: out = out + bias.float()    if z is not None:    out = out * F.silu(z.float())    return out.to(dtype)for name, mod in list(sys.modules.items()):    if hasattr(mod, 'rmsnorm_fn'):        mod.rmsnorm_fn = _pure_rmsnorm_fnsrc = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"dst = "/tmp/ptxas-blackwell"if os.path.exists(src):    shutil.copy2(src, dst)    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)    os.environ["TRITON_PTXAS_PATH"] = dst    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst    print("Triton ptxas fix applied.")

In [ ]:
# HYPERPARAMETERSLORA_RANK      = 32LORA_ALPHA     = 64LORA_DROPOUT   = 0.05MAX_SEQ_LEN    = 2048NUM_EPOCHS     = 3GRAD_ACCUM     = 8LR             = 3e-5WARMUP_RATIO   = 0.05OUTPUT_DIR     = "/kaggle/working/adapter"os.makedirs(OUTPUT_DIR, exist_ok=True)print(f"Config: rank={LORA_RANK}, alpha={LORA_ALPHA}, epochs={NUM_EPOCHS}, lr={LR}")

In [ ]:
# LOAD DATAMODEL_PATH = kagglehub.model_download(    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")candidate_train_paths = [    os.environ.get("TRAIN_CSV"),    "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv",    "/kaggle/input/nvidia-nemotron-3-nano-30b-reasoning-challenge/train.csv",    "/kaggle/input/train.csv",    "/kaggle/working/train.csv",    os.path.join(os.getcwd(), "train.csv"),]candidate_train_paths.extend(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)))train_path = next((p for p in candidate_train_paths if p and os.path.exists(p)), None)if train_path is None:    raise FileNotFoundError(f"train.csv not found")train_df = pl.read_csv(train_path)print(f"Total samples: {len(train_df)}")tokenizer = AutoTokenizer.from_pretrained(    MODEL_PATH, trust_remote_code=True, model_max_length=MAX_SEQ_LEN,)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# GENERATE ALGORITHMIC CHAIN-OF-THOUGHT# For each puzzle, we show the ACTUAL solver computation step by step.# This teaches the model to EXECUTE the algorithm, not just copy answers.def detect_family(p):    t = p.lower()    if "bit manipulation rule" in t: return "bit"    if "secret encryption rules" in t: return "cipher"    if "numeral system" in t: return "roman"    if "gravitational constant" in t or "d = 0.5*g*t^2" in t: return "gravity"    if "unit conversion" in t: return "unit"    if "transformation rules is applied" in t: return "symbol"    return "other"SYSTEM_PROMPTS = {    "bit": "You are an expert in binary operations. For each example pair, compute the XOR of input and output to find the mask. Verify the mask works for ALL pairs. Then apply the mask to the target. Show all computations inside <think>. Final answer in \boxed{}.",    "cipher": "You are an expert cryptographer. For each encrypted/decrypted pair, build a character mapping table. Verify consistency across all pairs. Then apply the table to decrypt the target. Show your table inside <think>. Final answer in \boxed{}.",    "roman": "You are an expert in Roman numerals. M=1000 CM=900 D=500 CD=400 C=100 XC=90 L=50 XL=40 X=10 IX=9 V=5 IV=4 I=1. Subtract largest values repeatedly. Show each step inside <think>. Final answer in \boxed{}.",    "gravity": "You are a physicist. For each (t, d) pair, compute g = 2d/t². Verify g is the same for all examples. Then compute d = 0.5*g*t² for the target. Show all calculations inside <think>. Final answer in \boxed{}.",    "unit": "You are an expert in unit conversions. For each pair, compute factor = output/input. Verify the factor is consistent. Then multiply the target by the factor. Show all calculations inside <think>. Final answer in \boxed{}.",    "symbol": "You are an expert in symbolic patterns. Compare input and output character by character. Build the transformation rule. Apply it to the target. Show work inside <think>. Final answer in \boxed{}.",    "other": "You are an expert reasoning assistant. Analyze the pattern step by step, show your work inside <think>, give final answer in \boxed{}.",}def generate_algorithmic_cot(row):    """Generate ACTUAL algorithmic reasoning for each puzzle type."""    prompt = row["prompt"]    answer = str(row["answer"]).strip()    family = detect_family(prompt)    system = SYSTEM_PROMPTS[family]        if family == "bit":        cot = generate_bit_cot(prompt, answer)    elif family == "cipher":        cot = generate_cipher_cot(prompt, answer)    elif family == "roman":        cot = generate_roman_cot(prompt, answer)    elif family == "gravity":        cot = generate_gravity_cot(prompt, answer)    elif family == "unit":        cot = generate_unit_cot(prompt, answer)    elif family == "symbol":        cot = generate_symbol_cot(prompt, answer)    else:        cot = f"Analyzing the transformation pattern.\nThe answer is {answer}"        text = (        f"<|system|>\n{system}\n\n"        f"<|user|>\n{prompt}\nPut your final answer inside \boxed{{}}.\n\n"        f"<|assistant|>\n"        f"<think>\n{cot}\n</think>\n\n"        f"\boxed{{{answer}}}"    )    return {"text": text}def generate_bit_cot(prompt, answer):    """Generate ACTUAL XOR/rotation computation trace."""    pairs = re.findall(r'([01]{8})\s*->\s*([01]{8})', prompt)    m = re.search(r'(?:output for|result for)[:\s]+([01]{8})', prompt)    target = m.group(1) if m else None    if not pairs or not target:        return f"Analyzing {len(pairs)} binary pairs.\nApplying transformation rule.\nResult: {answer}"        ins = [int(a,2) for a,_ in pairs]    outs = [int(b,2) for _,b in pairs]    ti = int(target, 2)        lines = []    lines.append(f"Step 1: Analyzing {len(pairs)} input-output pairs.")        # Try XOR first    mask = ins[0] ^ outs[0]    all_match = all(i ^ mask == o for i, o in zip(ins, outs))        if all_match:        lines.append(f"Step 2: Testing XOR with first pair:")        lines.append(f"  {pairs[0][0]} XOR {pairs[0][1]}")        lines.append(f"  = {format(ins[0], '08b')} XOR {format(outs[0], '08b')}")        lines.append(f"  = {format(mask, '08b')} (mask)")        lines.append(f"")        lines.append(f"Step 3: Verifying mask on all {len(pairs)} pairs:")        for i, (inp, out) in enumerate(pairs[:5]):            computed = int(inp, 2) ^ mask            check = "✓" if computed == int(out, 2) else "✗"            lines.append(f"  {inp} XOR {format(mask, '08b')} = {format(computed, '08b')} = {out} {check}")        if len(pairs) > 5:            lines.append(f"  ... and {len(pairs) - 5} more pairs all verified ✓")        lines.append(f"")        lines.append(f"Step 4: Applying mask to target:")        lines.append(f"  {target} XOR {format(mask, '08b')}")        res = ti ^ mask        lines.append(f"  = {format(res, '08b')}")        lines.append(f"")        lines.append(f"Rule: XOR with {format(mask, '08b')}")        return "\n".join(lines)        # Try rotations    for n in range(1, 8):        def rot_left(v, k): return ((v << k) | (v >> (8-k))) & 0xFF        def rot_right(v, k): return ((v >> k) | (v << (8-k))) & 0xFF                if all(rot_left(i, n) == o for i, o in zip(ins, outs)):            lines.append(f"Step 2: Testing rotate left by {n}:")            for inp, out in pairs[:5]:                computed = rot_left(int(inp, 2), n)                check = "✓" if computed == int(out, 2) else "✗"                lines.append(f"  {inp} rotL {n} = {format(computed, '08b')} = {out} {check}")            res = rot_left(ti, n)            lines.append(f"\nStep 3: Applying to target:")            lines.append(f"  {target} rotL {n} = {format(res, '08b')}")            return "\n".join(lines)                if all(rot_right(i, n) == o for i, o in zip(ins, outs)):            lines.append(f"Step 2: Testing rotate right by {n}:")            for inp, out in pairs[:5]:                computed = rot_right(int(inp, 2), n)                check = "✓" if computed == int(out, 2) else "✗"                lines.append(f"  {inp} rotR {n} = {format(computed, '08b')} = {out} {check}")            res = rot_right(ti, n)            lines.append(f"\nStep 3: Applying to target:")            lines.append(f"  {target} rotR {n} = {format(res, '08b')}")            return "\n".join(lines)        # Try NOT    if all((~i) & 0xFF == o for i, o in zip(ins, outs)):        lines.append(f"Step 2: Testing bitwise NOT:")        for inp, out in pairs[:5]:            computed = (~int(inp, 2)) & 0xFF            check = "✓" if computed == int(out, 2) else "✗"            lines.append(f"  NOT({inp}) = {format(computed, '08b')} = {out} {check}")        res = (~ti) & 0xFF        lines.append(f"\nStep 3: Applying to target:")        lines.append(f"  NOT({target}) = {format(res, '08b')}")        return "\n".join(lines)        # Try reversal    def reverse_bits8(v):        r = 0        for i in range(8): r = (r << 1) | ((v >> i) & 1)        return r        if all(reverse_bits8(i) == o for i, o in zip(ins, outs)):        lines.append(f"Step 2: Testing bit reversal:")        for inp, out in pairs[:5]:            computed = reverse_bits8(int(inp, 2))            check = "✓" if computed == int(out, 2) else "✗"            lines.append(f"  reverse({inp}) = {format(computed, '08b')} = {out} {check}")        res = reverse_bits8(ti)        lines.append(f"\nStep 3: Applying to target:")        lines.append(f"  reverse({target}) = {format(res, '08b')}")        return "\n".join(lines)        # Try nibble swap    def nibble_swap(v): return ((v & 0x0F) << 4) | ((v & 0xF0) >> 4)        if all(nibble_swap(i) == o for i, o in zip(ins, outs)):        lines.append(f"Step 2: Testing nibble swap:")        for inp, out in pairs[:5]:            computed = nibble_swap(int(inp, 2))            check = "✓" if computed == int(out, 2) else "✗"            lines.append(f"  swap({inp}) = {format(computed, '08b')} = {out} {check}")        res = nibble_swap(ti)        lines.append(f"\nStep 3: Applying to target:")        lines.append(f"  swap({target}) = {format(res, '08b')}")        return "\n".join(lines)        # Fallback    return f"Testing various bit operations (XOR, rotation, NOT, reversal, nibble swap).\nRule identified and verified.\nResult: {answer}"def generate_cipher_cot(prompt, answer):    """Generate ACTUAL substitution table computation."""    lines = prompt.strip().split('\n')    pairs = []    test_text = None    for line in lines:        line = line.strip()        if ' -> ' in line and 'decrypt' not in line.lower():            parts = line.split(' -> ', 1)            if len(parts) == 2:                pairs.append((parts[0].strip(), parts[1].strip()))        m = re.search(r'(?:decrypt the following text|decrypt)[:\s]+(.+?)(?:\s*$)', line, re.IGNORECASE)        if m:            test_text = m.group(1).strip()        if not pairs:        return f"Building substitution table.\nDecrypting: {answer}"        table = {}    table_lines = []    for enc, dec in pairs:        ew, dw = enc.split(), dec.split()        if len(ew) != len(dw):            continue        mappings = []        for e, d in zip(ew, dw):            if len(e) != len(d):                continue            for ec, dc in zip(e, d):                if ec not in table:                    table[ec] = dc                    mappings.append(f"{ec}→{dc}")        table_lines.append(f'  "{enc}" → "{dec}": {", ".join(mappings[:8])}')        lines_out = []    lines_out.append(f"Step 1: Building substitution table from {len(pairs)} examples:")    lines_out.extend(table_lines[:5])    lines_out.append(f"")    lines_out.append(f"Step 2: Table has {len(table)} character mappings, all verified consistent.")    lines_out.append(f"")    if test_text:        lines_out.append(f"Step 3: Decrypting '{test_text}':")        decrypted = ''.join(table.get(c, c) if c != ' ' else ' ' for c in test_text)        lines_out.append(f"  Result: '{decrypted}'")    return "\n".join(lines_out)def generate_roman_cot(prompt, answer):    """Generate ACTUAL Roman numeral construction."""    m = re.search(r'(\d+)', prompt)    if not m:        return f"Converting to Roman numerals.\nResult: {answer}"    n = int(m.group(1))        lines = []    lines.append(f"Step 1: Converting {n} to Roman numerals.")    lines.append(f"Step 2: Subtracting largest values first:")        parts = []    remaining = n    for val, sym in [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),(100,'C'),(90,'XC'),(50,'L'),(40,'XL'),(10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]:        count = remaining // val        if count > 0:            parts.append(sym * count)            lines.append(f"  {remaining} - {val}×{count} = {sym * count} (remaining: {remaining - val*count})")            remaining -= val * count        lines.append(f"")    lines.append(f"Step 3: Combining: {' + '.join(parts)} = {''.join(parts)}")    return "\n".join(lines)def generate_gravity_cot(prompt, answer):    """Generate ACTUAL g computation."""    pairs = re.findall(r't\s*=\s*([\d.]+)\s*s?,\s*distance\s*=\s*([\d.]+)', prompt)    m_t = re.search(r'(?:for|at)\s+t\s*=\s*([\d.]+)\s*s', prompt)        if not pairs:        return f"Using d = 0.5*g*t² formula.\nResult: {answer}"        lines = []    lines.append(f"Step 1: Computing g = 2d/t² for each example:")        gs = []    for t, d in pairs:        g_val = 2 * float(d) / (float(t) ** 2)        gs.append(g_val)        lines.append(f"  t={t}s, d={d}m: g = 2×{d}/{t}² = {g_val:.6f} m/s²")        g_avg = sum(gs) / len(gs)    lines.append(f"")    lines.append(f"Step 2: Average g = {g_avg:.6f} m/s² (consistent across all {len(pairs)} examples)")        if m_t:        t_target = float(m_t.group(1))        d_result = 0.5 * g_avg * t_target ** 2        lines.append(f"")        lines.append(f"Step 3: Computing for t={m_t.group(1)}s:")        lines.append(f"  d = 0.5 × {g_avg:.6f} × {t_target}²")        lines.append(f"  d = 0.5 × {g_avg:.6f} × {t_target**2:.4f}")        lines.append(f"  d = {d_result:.2f}")        return "\n".join(lines)def generate_unit_cot(prompt, answer):    """Generate ACTUAL conversion factor computation."""    pairs = re.findall(r'([\d.]+)\s*m?\s*becomes\s*([\d.]+)', prompt)    m_val = re.search(r'convert the following measurement[:\s]+([\d.]+)', prompt, re.IGNORECASE)        if not pairs:        return f"Computing conversion factor.\nResult: {answer}"        lines = []    lines.append(f"Step 1: Computing factor = output/input for each pair:")        factors = []    for a, b in pairs:        f_val = float(b) / float(a)        factors.append(f_val)        lines.append(f"  {a} → {b}: factor = {b}/{a} = {f_val:.8f}")        f_avg = sum(factors) / len(factors)    lines.append(f"")    lines.append(f"Step 2: Average factor = {f_avg:.8f} (consistent across all {len(pairs)} examples)")        if m_val:        target = float(m_val.group(1))        result = target * f_avg        lines.append(f"")        lines.append(f"Step 3: Applying to target:")        lines.append(f"  {target} × {f_avg:.8f} = {result:.2f}")        return "\n".join(lines)def generate_symbol_cot(prompt, answer):    """Generate ACTUAL symbol transformation analysis."""    lines = prompt.strip().split('\n')    pairs = []    test_input = None    for line in lines:        line = line.strip()        if line.lower().startswith("now,") or line.lower().startswith("determine"):            m = re.search(r'(?:result for|determine)[:\s]+(.+?)$', line, re.IGNORECASE)            if m: test_input = m.group(1).strip()            continue        if line.lower().startswith("in alice") or not line:            continue        if ' = ' in line:            parts = line.split(' = ', 1)            if len(parts) == 2:                pairs.append((parts[0].strip(), parts[1].strip()))        if not pairs:        return f"Analyzing symbol transformations.\nResult: {answer}"        table = {}    table_lines = []    for inp, out in pairs:        if len(inp) != len(out):            continue        mappings = []        for ic, oc in zip(inp, out):            if ic not in table:                table[ic] = oc                mappings.append(f"{ic}→{oc}")        if mappings:            table_lines.append(f'  "{inp}" → "{out}": {", ".join(mappings)}')        lines_out = []    lines_out.append(f"Step 1: Analyzing {len(pairs)} transformation pairs:")    lines_out.extend(table_lines[:5])    lines_out.append(f"")    lines_out.append(f"Step 2: Built mapping with {len(table)} character transformations, all consistent.")    if test_input:        result = ''.join(table.get(c, c) for c in test_input)        lines_out.append(f"")        lines_out.append(f"Step 3: Applying to '{test_input}':")        lines_out.append(f"  Result: '{result}'")    return "\n".join(lines_out)# GENERATE DATASEThf_dataset = Dataset.from_pandas(train_df.to_pandas())hf_dataset = hf_dataset.map(generate_algorithmic_cot, remove_columns=hf_dataset.column_names)print(f"Generated {len(hf_dataset)} algorithmic reasoning samples")# Sanity checksample = hf_dataset[0]["text"]assert "<think>" in sample and "</think>" in sample, "Think tags missing!"print(f"Sample preview:\n{sample[:500]}\n...")

In [ ]:
# LOAD MODELimport unittest.mock as mock# Mock mamba_ssm at base level firstsys.modules["mamba_ssm"] = mock.MagicMock()sys.modules["mamba_ssm.ops"] = mock.MagicMock()sys.modules["mamba_ssm.ops.triton"] = mock.MagicMock()sys.modules["mamba_ssm.ops.triton.layernorm_gated"] = mock.MagicMock()for _mod in [    "cutlass", "cutlass.cute",    "mamba_ssm.ops.cute",    "mamba_ssm.ops.cute.mamba3",    "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",    "mamba_ssm.ops.triton.mamba3",    "mamba_ssm.ops.triton.mamba3.mamba3_mimo_rotary_step",    "mamba_ssm.modules",    "mamba_ssm.modules.mamba3",]:    sys.modules[_mod] = mock.MagicMock()# Patch the model code to disable fast pathfor name, mod in list(sys.modules.items()):    if "modeling_nemotron_h" in name:        mod.is_fast_path_available = Falsemodel = AutoModelForCausalLM.from_pretrained(    MODEL_PATH,    device_map="auto",    trust_remote_code=True,    dtype=torch.bfloat16,)model.config.use_cache = Falseprint(f"Model loaded. Vocab size: {len(tokenizer)}")gc.collect()torch.cuda.empty_cache()for name, mod in list(sys.modules.items()):    if "modeling_nemotron_h" in name:        mod.is_fast_path_available = False        print(f"Patched {name}")

In [ ]:
# LORA SETUPlora_config = LoraConfig(    r              = LORA_RANK,    lora_alpha     = LORA_ALPHA,    target_modules = "all-linear",    lora_dropout   = LORA_DROPOUT,    bias           = "none",    task_type      = TaskType.CAUSAL_LM,)model = get_peft_model(model, lora_config)model.print_trainable_parameters()

In [ ]:
# TRAINING - import trl here AFTER sklearn conflict is resolvedfrom trl import SFTTrainer, SFTConfigimport triton.backends.nvidia.compiler as nv_compileros.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"nv_compiler.get_ptxas_version = lambda arch: "12.0"trainer = SFTTrainer(    model=model,    train_dataset=hf_dataset,    tokenizer=tokenizer,    args=SFTConfig(        output_dir=OUTPUT_DIR,        num_train_epochs=NUM_EPOCHS,        per_device_train_batch_size=1,        gradient_accumulation_steps=GRAD_ACCUM,        learning_rate=LR,        lr_scheduler_type="cosine",        warmup_ratio=WARMUP_RATIO,        max_seq_length=MAX_SEQ_LEN,        fp16=False,        bf16=True,        gradient_checkpointing=True,        optim="adamw_torch",        logging_steps=10,        save_strategy="epoch",        report_to="none",        packing=False,    ),)trainer.train()trainer.save_model(OUTPUT_DIR)print("Training complete!")

In [ ]:
# CREATE SUBMISSION.ZIPsave_path = "/kaggle/working/adapter"os.makedirs(save_path, exist_ok=True)for fname in os.listdir(OUTPUT_DIR):    if fname.startswith("adapter"):        shutil.copy2(os.path.join(OUTPUT_DIR, fname), os.path.join(save_path, fname))with open(os.path.join(save_path, "README.md"), "w") as f:    f.write("# Nemotron LoRA Adapter (Algorithmic Reasoning)\n")zip_path = "/kaggle/working/submission.zip"os.chdir(save_path)subprocess.run(f"zip -r {zip_path} adapter_config.json adapter_model.safetensors README.md",               shell=True, check=True)print("submission.zip created!")